In [10]:
df.to_csv("validation_results.csv", index=False)

print("Saved validation_results.csv")

Saved validation_results.csv


In [13]:
accuracy = df["Agreement"].mean() * 100

print(f"Directional Accuracy: {accuracy:.2f}%")

Directional Accuracy: 64.29%


In [12]:
df = pd.DataFrame(results)

print("Validated stocks:", len(df))

df.head()

Validated stocks: 14


,Ticker,Current Price,DCF Value,Analyst Target,Beta,Discount Rate,Model View,Analyst View,Agreement
0,AAPL,299.370,263.441898,312.71628,1.09,7.26,OVERVALUED,UNDERVALUED,False
1,MSFT,393.505,399.771520,561.38763,1.10,7.31,UNDERVALUED,UNDERVALUED,True
2,GOOGL,373.770,545.818431,432.83453,1.24,7.71,UNDERVALUED,UNDERVALUED,True
3,META,600.020,1404.048986,827.31830,1.23,7.69,UNDERVALUED,UNDERVALUED,True
4,NVDA,208.910,861.107966,298.93220,2.00,10.00,UNDERVALUED,UNDERVALUED,True


In [11]:
results = []

for ticker in stocks:

    try:

        print(f"Processing {ticker}...")

        data = get_stock_data(ticker)

        current_price = data["current_price"]

        # Growth assumption
        analyst_growth = data.get("analyst_growth")

        if analyst_growth is not None:
            growth_rate = analyst_growth
        else:
            growth_rate = data["growth_rate"]

        # Dynamic discount rate from beta
        beta = max(0.8, min(data.get("beta", 1.0), 2.0))
        discount_rate = 0.04 + beta * 0.03

        projected = project_cash_flows(
            data["fcf_per_share"],
            growth_rate,
            PROJECTION_YEARS
        )

        dcf = calculate_dcf(
            projected,
            discount_rate,
            TERMINAL_GROWTH,
            PROJECTION_YEARS
        )

        intrinsic_value = dcf["intrinsic_value"]

        info = yf.Ticker(ticker).info
        target_price = info.get("targetMeanPrice")

        if target_price is None:
            continue

        model_view = (
            "UNDERVALUED"
            if intrinsic_value > current_price
            else "OVERVALUED"
        )

        analyst_view = (
            "UNDERVALUED"
            if target_price > current_price
            else "OVERVALUED"
        )

        agreement = (
            model_view == analyst_view
        )

        results.append({

            "Ticker": ticker,
            "Current Price": current_price,
            "DCF Value": intrinsic_value,
            "Analyst Target": target_price,
            "Beta": round(beta, 2),
            "Discount Rate": round(discount_rate * 100, 2),
            "Model View": model_view,
            "Analyst View": analyst_view,
            "Agreement": agreement

        })

    except Exception as e:

        print(f"{ticker} failed: {e}")

Processing AAPL...
Processing MSFT...
Processing GOOGL...
Processing AMZN...
AMZN failed: Failed to compute historical FCF
Processing META...
Processing NVDA...
Processing TSLA...
Processing JPM...
JPM failed: Failed to compute historical FCF
Processing JNJ...
Processing PG...
Processing INFY.NS...
Processing TCS.NS...
Processing RELIANCE.NS...
RELIANCE.NS failed: Failed to compute historical FCF
Processing HDFCBANK.NS...
Processing WIPRO.NS...
Processing TATAMOTORS.NS...


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: TATAMOTORS.NS"}}}


TATAMOTORS.NS failed: Could not fetch price for TATAMOTORS.NS
Processing ICICIBANK.NS...
ICICIBANK.NS failed: Failed to compute historical FCF
Processing BAJFINANCE.NS...
BAJFINANCE.NS failed: Failed to compute historical FCF
Processing HINDUNILVR.NS...
Processing ITC.NS...


In [6]:
import yfinance as yf

test = yf.Ticker("MSFT")

print(test.info.get("targetMeanPrice"))

561.38763


In [5]:
results = []

In [4]:
stocks = [
    "AAPL",
    "MSFT",
    "GOOGL",
    "AMZN",
    "META",
    "NVDA",
    "TSLA",
    "JPM",
    "JNJ",
    "PG",
    "INFY.NS",
    "TCS.NS",
    "RELIANCE.NS",
    "HDFCBANK.NS",
    "WIPRO.NS",
    "TATAMOTORS.NS",
    "ICICIBANK.NS",
    "BAJFINANCE.NS",
    "HINDUNILVR.NS",
    "ITC.NS"
]

len(stocks)

20

In [3]:
import pandas as pd
import numpy as np


In [2]:

from dcf_engine import (get_stock_data,project_cash_flows,calculate_dcf,TERMINAL_GROWTH,PROJECTION_YEARS)